Code pour faire fonctionner pyspark (ne pas exécuter si pas besoin)

In [30]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [31]:
import sys
# Affiche le chemin de l'exécutable Python utilisé par ce notebook
print(sys.executable)

c:\Users\etien\miniconda3\envs\nf26\python.exe


In [32]:
import os
from pyspark.sql import SparkSession

# Arrêter la session Spark existante pour appliquer les nouveaux paramètres
if 'spark' in locals():
    spark.stop()

# Définir explicitement le chemin de l'exécutable Python pour Spark
# Ce chemin correspond à l'environnement du kernel du notebook
python_path = r'c:\Users\etien\miniconda3\envs\nf26\python.exe'
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

# Recréer la SparkSession avec la configuration mémoire et la configuration du worker
spark = (
    SparkSession.builder
    .appName("GHG-Inventory-ETL")
    .config("spark.driver.memory", "8g")
    .config("spark.python.worker.exec", python_path)
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print(f"PYSPARK_PYTHON est maintenant défini sur : {os.environ.get('PYSPARK_PYTHON')}")
print("Nouvelle session Spark créée avec succès.")

PYSPARK_PYTHON est maintenant défini sur : c:\Users\etien\miniconda3\envs\nf26\python.exe
Nouvelle session Spark créée avec succès.


In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *
from datetime import date, timedelta

## Partie ETL

#### Fonctions d'homogénéisation de la langue des fonctions et missions

In [34]:
def clean_langue_fonction(df, site):
    match site:
        case "BERLIN":
            map_fonctions = {
                "Ökonom": "Economist",
                "Führungskraft": "Business Executive",
                "Personalleiter": "HRD",
                "Computeringenieur": "Computer Engineer",
                "Dateningenieur": "Data Engineer",
            }
            df["FONCTION_PERSONNEL"] = df["FONCTION_PERSONNEL"].replace(map_fonctions)
        case "PARIS":
            map_fonctions = {
                "Ingénieur Informaticien": "Computer Engineer",
                "Ingénieur Data": "Data Engineer",
                "Economiste": "Economist",
                "DRH": "HRD",
                "Cadre": "Business Executive",
            }
            df["FONCTION_PERSONNEL"] = df["FONCTION_PERSONNEL"].replace(map_fonctions)


In [35]:
def clean_langue_mission(df, site):
    match site:
        case "BERLIN":
            map_type_mission = {
                "Geschäftstreffen": "Business Meeting",
                "Konferenz": "Conference",
                "Schulung": "Vocational Training",
                "Meeting": "Team Meeting",
                "Entwicklung": "Development",
            }
            df["TYPE_MISSION"] = df["TYPE_MISSION"].replace(map_type_mission)
        case "PARIS":
            map_type_mission = {
                "Conférence": "Conference",
                "Formation": "Vocational Training",
                "Réunion": "Team Meeting",
                "Rencontre entreprises": "Business Meeting",
                "Développement": "Development",
            }
            df["TYPE_MISSION"] = df["TYPE_MISSION"].replace(map_type_mission)

#### Fonction d'homogénéisation des fuseaux horaires

In [36]:
def clean_date(df, site, date_col):
    """
    Convertit la colonne de dates d'un site donné vers UTC+2 (heure de Paris).
    Les dates sont supposées être en heure locale du site (naive).
    """
    # Mapping site -> fuseau horaire IANA
    site_tz = {
        "PARIS":      "Europe/Paris",
        "BERLIN":     "Europe/Berlin",
        "LONDON":     "Europe/London",
        "NEWYORK":    "America/New_York",
        "LOSANGELES": "America/Los_Angeles",
        "SHANGHAI":   "Asia/Shanghai",
    }

    tz = site_tz[site]

    # S'assurer que la colonne est bien en datetime
    df[date_col] = pd.to_datetime(df[date_col])

    # Localiser la date dans le fuseau du site, puis convertir vers Paris (UTC+2 en été)
    df[date_col] = (
        df[date_col]
          .dt.tz_localize(tz, ambiguous=False, nonexistent='shift_forward') # on dit "cette heure est en heure locale du site"
          # le param ambiguous est utilisé pour les heures du changement d'heure
          .dt.tz_convert("Europe/Paris")  # on la convertit vers Paris
          .dt.tz_localize(None)           # on retire l'info de fuseau pour rester naive
    )

#### Chargement de l'ensemble des données du personnel

In [37]:
def extract_personnel():
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")

    personnel_dfs = []

    # On charge la liste du personnel de chaque site dans un dataframe
    for site in sites:
        file_path = base_path / f"BDD_BGES_{site}" / f"PERSONNEL_{site}.txt"
        df = pd.read_csv(str(file_path), sep=';')
        clean_langue_fonction(df, site) 
        df['ID_SITE'] = site
        
        personnel_dfs.append(df)

    # On combine les dataframes 
    personnel_df = pd.concat(personnel_dfs)
  
    # On sélectionne uniquement les colonnes nécessaires
    personnel_final_df = personnel_df[['ID_PERSONNEL','FONCTION_PERSONNEL', 'ID_SITE']]
    return personnel_final_df

In [38]:
personnel_df = extract_personnel()
personnel_df.head()

,ID_PERSONNEL,FONCTION_PERSONNEL,ID_SITE
0,KeyPers_Paris_1230000,Business Executive,PARIS
1,KeyPers_Paris_1230001,HRD,PARIS
2,KeyPers_Paris_1230002,Data Engineer,PARIS
3,KeyPers_Paris_1230003,Computer Engineer,PARIS
4,KeyPers_Paris_1230004,Computer Engineer,PARIS


#### Traitement journalier des données de matériel informatique

In [39]:
def extract_materiel(start_date, end_date):
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")
    # Liste pour stocker les dataframes de chaque jour
    all_materiel_dfs = []

    # Boucle sur chaque jour
    for single_date in pd.date_range(start_date, end_date):
        day_str = single_date.strftime("%Y%m%d")
        
        # Liste pour stocker les dataframes de matériel de la journée
        materiel_day_dfs = []
        
        # Boucle sur chaque site
        for site in sites:
            file_path = base_path / f"BDD_BGES_{site}/BDD_BGES_{site}_INFORMATIQUE/MATERIEL_INFORMATIQUE_{day_str}.txt"
            
            # Vérifier si le fichier existe avant de le lire
            if file_path.exists():
                df = pd.read_csv(str(file_path), sep=';')
                clean_date(df, site, "DATE_ACHAT")
                df['ID_DATE'] = df['DATE_ACHAT'].dt.date #pour jointure 
                materiel_day_dfs.append(df)
                
        # Si des fichiers ont été trouvés pour ce jour
        if materiel_day_dfs:
            # Combiner les données de tous les sites pour la journée et l'ajouter à la liste globale
            all_materiel_dfs.append(pd.concat(materiel_day_dfs, ignore_index=True))

    # Après la boucle, vérifier si on a collecté des données
    if all_materiel_dfs:
        # Combiner les données de tous les jours en un seul DataFrame
        materiel_total_df = pd.concat(all_materiel_dfs, ignore_index=True)
        return materiel_total_df
    else:
        print("Aucun fichier de matériel trouvé pour la période spécifiée.")

In [40]:
start_date = date(2026, 4, 29)
end_date = date(2026, 11, 14) 

mat_df = extract_materiel(start_date, end_date)
mat_df.head()

,ID_MATERIELINFO,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_ACHAT,TYPE,MODELE,ID_DATE
0,Paris_MATERIEL_INFO_202604290,KeyPers_Paris_1232362,Name2362,FistName2362,2026-04-29 08:17:31,PC fixe sans ecran,Z,2026-04-29
1,Paris_MATERIEL_INFO_202604291,KeyPers_Paris_1232165,Name2165,FistName2165,2026-04-29 09:42:55,imprimante,Laser A3 (>100kg),2026-04-29
2,Paris_MATERIEL_INFO_202604292,KeyPers_Paris_1231951,Name1951,FistName1951,2026-04-29 13:58:12,PC fixe sans ecran,Precision tower 3xxx,2026-04-29
3,Paris_MATERIEL_INFO_202604293,KeyPers_Paris_1230614,Name614,FistName614,2026-04-29 13:19:31,Telephone IP,,2026-04-29
4,Paris_MATERIEL_INFO_202604294,KeyPers_Paris_1232952,Name2952,FistName2952,2026-04-29 13:55:41,,modèle par défaut,2026-04-29


#### Traitement journalier des données de missions

In [41]:
# Même logique que la fonction d'extraction du matériel
def extract_missions(start_date, end_date):
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")
    
    all_missions_dfs = []

    for single_date in pd.date_range(start_date, end_date):
        day_str = single_date.strftime("%Y%m%d")
        
        missions_day_dfs = []
        
        for site in sites:
            file_path = base_path / f"BDD_BGES_{site}/BDD_BGES_{site}_MISSION/MISSION_{day_str}.txt"
            
            if file_path.exists():
                df = pd.read_csv(str(file_path), sep=';')
                clean_langue_mission(df, site)
                clean_date(df, site, "DATE_MISSION")
                df['ID_SITE'] = site
                df['ID_DATE'] = df['DATE_MISSION'].dt.date # pour jointure 
                missions_day_dfs.append(df)
                
        if missions_day_dfs:
            all_missions_dfs.append(pd.concat(missions_day_dfs, ignore_index=True))

    if all_missions_dfs :
        missions_total_df = pd.concat(all_missions_dfs, ignore_index=True)
        return missions_total_df
    else:
        print("Aucun fichier de missions trouvé pour la période spécifiée.")

In [42]:
missions_df = extract_missions(start_date, end_date)
missions_df.head()

C:\Users\etien\AppData\Local\Temp\ipykernel_13404\152790702.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])
C:\Users\etien\AppData\Local\Temp\ipykernel_13404\152790702.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])
C:\Users\etien\AppData\Local\Temp\ipykernel_13404\152790702.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])


,ID_MISSION,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_MISSION,TYPE_MISSION,VILLE_DEPART,PAYS_DEPART,VILLE_DESTINATION,PAYS_DESTINATION,TRANSPORT,ALLER_RETOUR,ID_SITE,ID_DATE
0,Paris_202604290,KeyPers_Paris_1233578,Name3578,FistName3578,2026-04-29 15:01:12,Business Meeting,Paris,France,New-York,USA,Avion,oui,PARIS,2026-04-29
1,Paris_202604291,KeyPers_Paris_1234968,Name4968,FistName4968,2026-04-29 18:41:44,Development,Paris,France,London,England,Avion,oui,PARIS,2026-04-29
2,Paris_202604292,KeyPers_Paris_1233110,Name3110,FistName3110,2026-04-29 09:13:25,Business Meeting,Paris,France,Washington,USA,Avion,oui,PARIS,2026-04-29
3,Paris_202604293,KeyPers_Paris_1230093,Name93,FistName93,2026-04-29 13:04:44,Conference,Paris,France,Berlin,Allemagne,Avion,oui,PARIS,2026-04-29
4,Paris_202604294,KeyPers_Paris_1234582,Name4582,FistName4582,2026-04-29 15:34:56,Development,Paris,France,Montreal,Canada,Avion,oui,PARIS,2026-04-29


Passage des données nettoyées au format Pyspark SQL Dataframe pour préparer leur chargement dans les tables finales (Load). 

In [43]:
sdf_materiel = spark.createDataFrame(mat_df)
sdf_personnel = spark.createDataFrame(personnel_df)
sdf_mission = spark.createDataFrame(missions_df)

#### Création tables de dimension

In [44]:
dim_materiel = (
    sdf_materiel
    .select("ID_MATERIELINFO", "TYPE", "MODELE")
    .withColumnRenamed("ID_MATERIELINFO", "ID_MATERIEL")
)

In [22]:
dim_materiel.show(5)

+--------------------+------------------+--------------------+
|         ID_MATERIEL|              TYPE|              MODELE|
+--------------------+------------------+--------------------+
|Paris_MATERIEL_IN...|PC fixe sans ecran|                   Z|
|Paris_MATERIEL_IN...|        imprimante|   Laser A3 (>100kg)|
|Paris_MATERIEL_IN...|PC fixe sans ecran|Precision tower 3xxx|
|Paris_MATERIEL_IN...|      Telephone IP|                    |
|Paris_MATERIEL_IN...|                  |   modèle par défaut|
+--------------------+------------------+--------------------+
only showing top 5 rows


In [45]:
data = [("BERLIN",), ("LONDON",), ("LOSANGELES",), ("NEWYORK",), ("PARIS",), ("SHANGHAI",)]
dim_site = spark.createDataFrame(data).toDF("ID_SITE")

In [24]:
dim_site.show(6)

+----------+
|   ID_SITE|
+----------+
|    BERLIN|
|    LONDON|
|LOSANGELES|
|   NEWYORK|
|     PARIS|
|  SHANGHAI|
+----------+



In [46]:
# Convertir toutes les dates au même format (date)
dates_missions = set(pd.to_datetime(missions_df['ID_DATE']))
dates_materiel = set(pd.to_datetime(mat_df['ID_DATE']))

# Combiner et trier
all_dates = sorted(dates_missions | dates_materiel)

# Créer la table de dimension
dim_date_data = [(d.date(),) for d in all_dates]
dim_date = spark.createDataFrame(dim_date_data, "ID_DATE date")

In [26]:
dim_date.show(5)

+----------+
|   ID_DATE|
+----------+
|2026-04-28|
|2026-04-29|
|2026-04-30|
|2026-05-01|
|2026-05-02|
+----------+
only showing top 5 rows


In [47]:
dim_personnel = sdf_personnel

In [22]:
dim_personnel.show(5)

+--------------------+------------------+-------+
|        ID_PERSONNEL|FONCTION_PERSONNEL|ID_SITE|
+--------------------+------------------+-------+
|KeyPers_Paris_123...|Business Executive|  PARIS|
|KeyPers_Paris_123...|               HRD|  PARIS|
|KeyPers_Paris_123...|     Data Engineer|  PARIS|
|KeyPers_Paris_123...| Computer Engineer|  PARIS|
|KeyPers_Paris_123...| Computer Engineer|  PARIS|
+--------------------+------------------+-------+
only showing top 5 rows


In [48]:
dim_mission = (
    sdf_mission
    .select(
        "ID_MISSION",
        "TYPE_MISSION",
        "VILLE_DEPART",
        "PAYS_DEPART",
        "VILLE_DESTINATION",
        "PAYS_DESTINATION",
        "TRANSPORT",
        "ALLER_RETOUR"
    )
)

In [23]:
dim_mission.show(5)

+---------------+----------------+------------+-----------+-----------------+----------------+---------+------------+
|     ID_MISSION|    TYPE_MISSION|VILLE_DEPART|PAYS_DEPART|VILLE_DESTINATION|PAYS_DESTINATION|TRANSPORT|ALLER_RETOUR|
+---------------+----------------+------------+-----------+-----------------+----------------+---------+------------+
|Paris_202604290|Business Meeting|       Paris|     France|         New-York|             USA|    Avion|         oui|
|Paris_202604291|     Development|       Paris|     France|           London|         England|    Avion|         oui|
|Paris_202604292|Business Meeting|       Paris|     France|       Washington|             USA|    Avion|         oui|
|Paris_202604293|      Conference|       Paris|     France|           Berlin|       Allemagne|    Avion|         oui|
|Paris_202604294|     Development|       Paris|     France|         Montreal|          Canada|    Avion|         oui|
+---------------+----------------+------------+---------

#### Création tables de faits

In [50]:
fait_materiel = (
    sdf_materiel
    .join(dim_date, "ID_DATE", "inner")
    .join(sdf_personnel, "ID_PERSONNEL", "inner")
    .join(dim_site, "ID_SITE", "inner")
    .select(col("ID_MATERIELINFO").alias("ID_MATERIEL"), "ID_PERSONNEL", "ID_SITE", col("ID_DATE").alias("ID_DATE_ACHAT"))
)

In [51]:
fait_materiel.show(5)

+--------------------+--------------------+-------+-------------+
|         ID_MATERIEL|        ID_PERSONNEL|ID_SITE|ID_DATE_ACHAT|
+--------------------+--------------------+-------+-------------+
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-23|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-23|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-23|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-22|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-22|
+--------------------+--------------------+-------+-------------+
only showing top 5 rows


In [52]:
fait_mission = (
    sdf_mission
    .join(dim_date, "ID_DATE", "inner")
    .join(sdf_personnel, "ID_PERSONNEL", "inner")
    .join(dim_site, "ID_SITE", "inner")
    .select("ID_MISSION", "ID_PERSONNEL", sdf_mission.ID_SITE, col("ID_DATE").alias("ID_DATE_MISSION"))
)

In [34]:
fait_mission.show(5)

+-----------------+--------------------+-------+---------------+
|       ID_MISSION|        ID_PERSONNEL|ID_SITE|ID_DATE_MISSION|
+-----------------+--------------------+-------+---------------+
|BERLIN_2026090627|KeyPers_Berlin_12...| BERLIN|     2026-09-06|
|BERLIN_2026052633|KeyPers_Berlin_12...| BERLIN|     2026-05-26|
| BERLIN_202605307|KeyPers_Berlin_12...| BERLIN|     2026-05-30|
| BERLIN_202605219|KeyPers_Berlin_12...| BERLIN|     2026-05-21|
|BERLIN_2026051616|KeyPers_Berlin_12...| BERLIN|     2026-05-16|
+-----------------+--------------------+-------+---------------+
only showing top 5 rows


## Réponse aux questions

Combien d’ingénieurs informaticiens travaillent sur le site de Paris ?

In [53]:
nb_inge_info = (
    dim_personnel
    .filter(col("ID_SITE") == "PARIS")
    .filter(col("FONCTION_PERSONNEL") == "Computer Engineer")
    .count()
)

print(f"{nb_inge_info} ingénieurs informaticiens travaillent sur le site de Paris")

1873 ingénieurs informaticiens travaillent sur le site de Paris


Combien d’ingénieurs Data travaillent sur les sites de London ?

In [54]:
nb_inge_data = (
    dim_personnel
    .filter(col("ID_SITE") == "LONDON")
    .filter(col("FONCTION_PERSONNEL") == "Data Engineer")
    .count()
)

print(f"{nb_inge_data} ingénieurs data travaillent sur le site de Londres")

1555 ingénieurs data travaillent sur le site de Londres


Combien de cadres travaillent dans l’organisation (tous sites compris) ?

In [55]:
nb_cadres = (
    dim_personnel
    .filter(col("FONCTION_PERSONNEL") == "Business Executive")
    .count()
)

print(f"{nb_cadres} cadres travaillent dans l'organisation")

3749 cadres travaillent dans l'organisation


Combien de PC portables ont été achetés par l’organisation entre mai et octobre 2024 ?

In [70]:
dim_materiel.filter(lower(col("TYPE")) == "pc portable").join(fait_materiel, "ID_MATERIEL").show()

+--------------------+-----------+--------------------+--------------------+----------+-------------+
|         ID_MATERIEL|       TYPE|              MODELE|        ID_PERSONNEL|   ID_SITE|ID_DATE_ACHAT|
+--------------------+-----------+--------------------+--------------------+----------+-------------+
|NewYork_MATERIEL_...|PC portable|        ZBook Studio|KeyPers_NewYork_1...|   NEWYORK|   2026-05-02|
|LONDON_MATERIEL_I...|PC portable|      EliteBook 10xx|KeyPers_London_12...|    LONDON|   2026-05-18|
|LA_MATERIEL_INFO_...|PC portable|Chromebook 14-15p...|  KeyPers_LA_1232013|LOSANGELES|   2026-05-10|
|Shanghai_MATERIEL...|PC portable|       EliteBook 6xx|KeyPers_Shanghai_...|  SHANGHAI|   2026-05-21|
|LA_MATERIEL_INFO_...|PC portable|                    |  KeyPers_LA_1232251|LOSANGELES|   2026-05-16|
|LONDON_MATERIEL_I...|PC portable|      Precision 3xxx|KeyPers_London_12...|    LONDON|   2026-05-04|
|LA_MATERIEL_INFO_...|PC portable| Moyenne 14/15pouces|  KeyPers_LA_1232129|LOSANG

In [ ]:
nb_pc_portables = (
    dim_materiel
    .filter(col("TYPE") == "PC portable")
    .join(fait_materiel, "ID_MATERIEL")
    .filter((col("ID_DATE_ACHAT") >= date(2024, 5, 1)) & (col("ID_DATE_ACHAT") <= date(2024, 10, 31)))
    .count()
)

print(f"{nb_pc_portables} PC portables ont été achetés entre mai et octobre 2024")

0 PC portables ont été achetés entre mai et octobre 2024


Quelle a été l’impact carbone des PC portables entre mai et octobre 2024 ?

In [63]:
impact_mat_info_psdf = ps.read_csv("./data/BDD_BGES/materiel_informatique_impact.csv")
impact_mat_info_sdf = impact_mat_info_psdf.to_spark()
impact_mat_info_sdf.show(5)

+------------------+-----------------+------+
|              Type|           Modèle|Impact|
+------------------+-----------------+------+
|PC fixe sans ecran|modèle par défaut|   350|
|PC fixe sans ecran|   Optiplex micro|   174|
|PC fixe sans ecran|   Optiplex small|   240|
|PC fixe sans ecran|   Optiplex tower|   260|
|PC fixe sans ecran| Wyse thin client|    69|
+------------------+-----------------+------+
only showing top 5 rows


c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [ ]:
# Vérifier les valeurs uniques de TYPE dans les données pandas
print("Valeurs uniques de TYPE :")
print(mat_df['TYPE'].unique())
print("\n")

# Vérification avec pandas - compter les PC portables
pc_portables_pandas = mat_df[
    (mat_df['TYPE'].str.lower() == 'pc portable') &
    (mat_df['ID_DATE'] >= pd.Timestamp('2026-05-01').date()) &
    (mat_df['ID_DATE'] <= pd.Timestamp('2026-10-31').date())
]

print(f"Nombre de PC portables (pandas) : {len(pc_portables_pandas)}")
print("\nExemples :")
print(pc_portables_pandas[['TYPE', 'MODELE', 'ID_DATE', 'ID_SITE']].head(10))